# Deep Learning 073 — How Self-Attention Works (Query, Key, Value)

Companion notebook to the lesson. We derive self-attention the way the lesson does:
build the obvious parameterless version first, **measure what is wrong with it**, and
let the measurements force Q/K/V into existence.

By the end you will have reproduced every number in the lesson:

| Claim | Number |
|---|---|
| Parameterless attention barely discriminates | max weight 0.350 vs uniform 0.278 |
| Its output is the sentence mean | cosine 0.9960 |
| Every word gets the same vector | cosine 0.9886 |
| The score matrix is symmetric | asymmetry exactly 0 |
| Untrained Q/K/V collapses **more** | 0.9992 |
| A static embedding cannot separate senses | cosine 1.000000, by definition |
| Trained Q/K/V can | cosine −0.9724 |
| One head is a low-rank restriction | rank 64 of 512 |

Only NumPy, scikit-learn and matplotlib. No TensorFlow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

np.random.seed(0)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def unit(v):
    return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-12)

def cos(a, b):
    return float(unit(a) @ unit(b))

## Part A — The three steps, by hand

Self-attention on a toy sentence, exactly as derived in the lesson: **dot product,
softmax, weighted sum.** Run it on three made-up 3-dimensional embeddings first, so
you can see the shapes before anything real is involved.

In [ ]:
words = ['money', 'bank', 'grows']
X = np.array([[0.9, 0.1, 0.2],     # money
              [0.7, 0.3, 0.1],     # bank
              [0.1, 0.8, 0.6]])    # grows

scores  = X @ X.T          # step 1 - every word against every word
weights = softmax(scores)  # step 2 - each row sums to 1
Y       = weights @ X      # step 3 - weighted sum of the embeddings

print('scores\n', np.round(scores, 3))
print('\nweights (rows sum to 1)\n', np.round(weights, 3))
print('\nrow sums:', weights.sum(1))
print('\ncontextual embeddings\n', np.round(Y, 3))

Three matrix operations, no loop over words. That is the parallelism the whole
architecture was built to get — and notice that **nothing in those three lines knows
what order the words arrived in.** Both facts come from the same place.

### Check the two structural flaws

Flaw one is a two-line proof. `(X @ X.T).T == X @ X.T` for *any* `X` whatsoever.

In [ ]:
assert np.allclose(scores, scores.T)
print('asymmetry, parameterless:', np.abs(scores - scores.T).max())

# Which means: bank attends to money exactly as much as money attends to bank.
i, j = words.index('bank'), words.index('money')
print(f'weight({words[i]} -> {words[j]}) = {weights[i, j]:.4f}')
print(f'weight({words[j]} -> {words[i]}) = {weights[j, i]:.4f}')
print('\nLanguage is not symmetric. An adjective depends on its noun far more')
print('than the noun depends on the adjective. This shape cannot express that.')

Flaw two: **count the parameters.** There are none. Dot product, softmax, weighted
sum — not one trainable weight. So there is nothing to fit to a task, and *a piece of
cake* will be blended into something about baked goods no matter how much translation
data you have.

## Part B — The collapse, on real embeddings

Toy vectors prove nothing about magnitudes. Build **real** embeddings: PPMI over a
word co-occurrence matrix from the 20 newsgroups corpus, reduced with an SVD, unit
normalised. This is the same construction the lesson used.

The download is cached after the first run.

In [ ]:
docs = fetch_20newsgroups(subset='train',
                          remove=('headers', 'footers', 'quotes')).data
counts = CountVectorizer(max_features=4000, stop_words='english', min_df=5)
X_counts = counts.fit_transform(docs)
vocab = np.array(counts.get_feature_names_out())
index = {w: i for i, w in enumerate(vocab)}

binary = (X_counts > 0).astype(np.float32)
cooccur = (binary.T @ binary).toarray()
np.fill_diagonal(cooccur, 0)
total = cooccur.sum()
marginal = cooccur.sum(1) / total
with np.errstate(divide='ignore', invalid='ignore'):
    ppmi = np.maximum(0, np.log((cooccur / total) /
                                np.outer(marginal, marginal) + 1e-12))
U, S, _ = np.linalg.svd(ppmi, full_matrices=False)
E = unit(U[:, :60] * S[:60])
D_EMB = E.shape[1]
print(f'{len(vocab)} words, {D_EMB} dimensions, unit norm')

In [ ]:
def collapse_stats(value_rows, scores):
    n = len(value_rows)
    W = softmax(scores)
    out = W @ value_rows
    pairs = [cos(out[i], out[j]) for i in range(n) for j in range(i + 1, n)]
    return dict(max_weight=float(W.max(1).mean()),
                uniform=1.0 / n,
                to_mean=float(np.mean([cos(out[i], value_rows.mean(0))
                                       for i in range(n)])),
                pairwise=float(np.mean(pairs)),
                asymmetry=float(np.abs(scores - scores.T).max()))

CANDIDATES = [['money', 'bank', 'account', 'interest'],
              ['river', 'bank', 'water', 'boat'],
              ['disk', 'drive', 'controller', 'memory'],
              ['car', 'drive', 'engine', 'road']]
sentences = [[w for w in c if w in index] for c in CANDIDATES]
sentences = [s for s in sentences if len(s) >= 3]
print('sentences:', '; '.join(' '.join(s) for s in sentences))

plain = []
for s in sentences:
    Es = E[[index[w] for w in s]]
    plain.append(collapse_stats(Es, Es @ Es.T))

for key, label in [('max_weight', 'mean largest attention weight'),
                   ('uniform',    'what uniform would be'),
                   ('to_mean',    'cosine(output, sentence mean)'),
                   ('pairwise',   'cosine between two outputs'),
                   ('asymmetry',  'max |S - S.T|')]:
    print(f'{label:<34}{np.mean([r[key] for r in plain]):.4f}')

**Read those five numbers together.**

Unit-norm embeddings put every dot product in `[-1, 1]`, so a softmax over a handful
of words is nearly flat. A nearly flat weighting *is* an average. And every word in
the sentence is averaging the same set of vectors — so every word receives nearly the
same output.

This is not a weak contextual embedding. It is **one embedding per sentence.**

Look at it directly: plot the attention matrix and see how little structure there is.

In [ ]:
s = sentences[0]
Es = E[[index[w] for w in s]]
W = softmax(Es @ Es.T)

fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(W, cmap='Blues', vmin=0, vmax=0.5)
ax.set_xticks(range(len(s)), s, rotation=45, ha='right')
ax.set_yticks(range(len(s)), s)
ax.set_title(f'parameterless attention\n(uniform would be {1/len(s):.3f})')
for i in range(len(s)):
    for j in range(len(s)):
        ax.text(j, i, f'{W[i, j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(im, shrink=0.8)
plt.tight_layout(); plt.show()

> **Try it.** Replace `unit(...)` in the embedding cell with the raw
> `U[:, :60] * S[:60]` and re-run. Without unit normalisation the norms are large,
> the dot products are large, and you get the *opposite* failure — a softmax pinned
> to the diagonal, because `e·e = ||e||²` is the largest entry in its row by
> Cauchy–Schwarz. Both extremes are broken. That tension is exactly what lesson 074
> is about.

## Part C — Q, K, V, and a result that comes out backwards

Each embedding is silently doing three jobs: it **asks** (query), it **is asked**
(key), and it **contributes content** (value). One vector cannot be optimal at three
different jobs, so derive three from it with three learned matrices.

Now the honest experiment. Build the Q/K/V version with **random, untrained**
matrices and re-measure the collapse. Predict the result before you run it.

In [ ]:
Wq0, Wk0, Wv0 = (np.random.randn(D_EMB, D_EMB) / np.sqrt(D_EMB) for _ in range(3))

rand_qkv = []
for s in sentences:
    Es = E[[index[w] for w in s]]
    rand_qkv.append(collapse_stats(Es @ Wv0, (Es @ Wq0) @ (Es @ Wk0).T))

print(f"{'':34}{'parameterless':>15}{'untrained QKV':>15}")
for key, label in [('max_weight', 'mean largest attention weight'),
                   ('pairwise',   'cosine between two outputs'),
                   ('asymmetry',  'max |S - S.T|')]:
    a = np.mean([r[key] for r in plain])
    b = np.mean([r[key] for r in rand_qkv])
    print(f'{label:<34}{a:>15.4f}{b:>15.4f}')

**The collapse got worse, not better.** 0.9992 against 0.9886.

The cause is not mysterious: random matrices scaled by `1/sqrt(d)` applied to already
near-parallel unit vectors produce an even *narrower* spread of scores, so the softmax
is flatter still.

Which sharpens what Q/K/V actually contributes. Structurally it buys exactly two
things:

1. a score matrix that is **no longer symmetric**, and
2. parameters that **have a gradient**.

Neither is an improvement in output quality by itself. Every quality claim for Q/K/V
rests on *training* — so let's train it.

## Part D — Training it on a real sense-disambiguation task

Lesson 072 measured that *drive* has two clear senses in this corpus and that its
static vector is captured outright by the storage sense. Here is the same word on an
actual task:

> Given a document containing **drive**, is it from the car newsgroup or the
> PC-hardware one?

Labels come from the corpus metadata, **not from any word list** — so the task is not
rigged by construction. Three representations of *drive* feed the same linear
classifier.

In [ ]:
CATS = ['comp.sys.ibm.pc.hardware', 'rec.autos']
TARGET, WINDOW = 'drive', 6
task = fetch_20newsgroups(subset='all', categories=CATS,
                          remove=('headers', 'footers', 'quotes'))
analyzer = counts.build_analyzer()

examples = []
for text, label in zip(task.data, task.target):
    toks = [t for t in analyzer(text) if t in index]
    if TARGET not in toks:
        continue
    t = toks.index(TARGET)
    ctx = toks[max(0, t - WINDOW): t + WINDOW + 1]
    if len(ctx) < 4:
        continue
    examples.append((np.array([index[w] for w in ctx]),
                     ctx.index(TARGET), int(label), ctx))

rng = np.random.default_rng(1)
order = rng.permutation(len(examples))
split = int(0.75 * len(order))
train_idx, test_idx = order[:split], order[split:]
y_all = np.array([e[2] for e in examples])
majority = max(np.bincount(y_all[test_idx])) / len(test_idx)

print(f'{len(examples)} examples ({len(train_idx)} train / {len(test_idx)} test)')
print(f'balance: {np.bincount(y_all)[0]} hardware / {np.bincount(y_all)[1]} autos')
print(f'majority-class accuracy on the test split: {majority:.1%}')

### The forward and backward pass, written out

One query position (the word *drive*), attending over its context window. The
backward pass is worth reading line by line — it is the whole of attention's
gradient in nine lines, and `ds = a * (da - a @ da)` is the softmax Jacobian
`diag(a) - a aᵀ` applied to a vector.

In [ ]:
DK = 32
scale = np.sqrt(DK)

def forward(Wq, Wk, Wv, Wo, b, ids, pos):
    Xs = E[ids]
    q, K, V = Xs[pos] @ Wq, Xs @ Wk, Xs @ Wv
    a = softmax((K @ q) / scale)
    y = a @ V
    return Xs, q, K, V, a, y, softmax(y @ Wo + b)

def train_attention(epochs=60, lr=0.25):
    Wq = np.random.randn(D_EMB, DK) / np.sqrt(D_EMB)
    Wk = np.random.randn(D_EMB, DK) / np.sqrt(D_EMB)
    Wv = np.random.randn(D_EMB, DK) / np.sqrt(D_EMB)
    Wo = np.random.randn(DK, 2) / np.sqrt(DK)
    b = np.zeros(2)
    for _ in range(epochs):
        for i in rng.permutation(train_idx):
            ids, pos, label, _ = examples[i]
            Xs, q, K, V, a, y, p = forward(Wq, Wk, Wv, Wo, b, ids, pos)
            dl = p.copy(); dl[label] -= 1.0        # softmax + cross-entropy
            dy = Wo @ dl
            da = V @ dy
            ds = a * (da - a @ da) / scale         # softmax Jacobian
            Wo -= lr * np.outer(y, dl); b -= lr * dl
            Wv -= lr * (Xs.T @ np.outer(a, dy))
            Wk -= lr * (Xs.T @ np.outer(ds, q))
            Wq -= lr * np.outer(Xs[pos], K.T @ ds)
    return Wq, Wk, Wv, Wo, b

Wq, Wk, Wv, Wo, b = train_attention()
print('trained')

In [ ]:
def evaluate_attention():
    correct, vecs = 0, {0: [], 1: []}
    for i in test_idx:
        ids, pos, label, _ = examples[i]
        *_, y, p = forward(Wq, Wk, Wv, Wo, b, ids, pos)
        correct += int(np.argmax(p) == label)
        vecs[label].append(y)
    return correct, cos(np.mean(vecs[0], 0), np.mean(vecs[1], 0))

def train_linear_on(feature_fn, dim=D_EMB, epochs=60, lr=0.25):
    Wl = np.random.randn(dim, 2) / np.sqrt(dim)
    bl = np.zeros(2)
    for _ in range(epochs):
        for i in rng.permutation(train_idx):
            ids, pos, label, _ = examples[i]
            f = feature_fn(ids, pos)
            p = softmax(f @ Wl + bl)
            dl = p.copy(); dl[label] -= 1.0
            Wl -= lr * np.outer(f, dl); bl -= lr * dl
    correct, vecs = 0, {0: [], 1: []}
    for i in test_idx:
        ids, pos, label, _ = examples[i]
        f = feature_fn(ids, pos)
        correct += int(np.argmax(softmax(f @ Wl + bl)) == label)
        vecs[label].append(f)
    return correct, cos(np.mean(vecs[0], 0), np.mean(vecs[1], 0))

def parameterless(ids, pos):
    Xs = E[ids]
    return softmax(Xs @ Xs[pos]) @ Xs

NT = len(test_idx)
static_n, static_sep = train_linear_on(lambda ids, pos: E[ids[pos]])
plain_n,  plain_sep  = train_linear_on(parameterless)
attn_n,   attn_sep   = evaluate_attention()

print(f"{'representation of the word drive':<40}{'correct':>9}{'acc':>8}{'sense cosine':>15}")
for name, n, sep in [('static embedding (one vector, always)', static_n, static_sep),
                     ('parameterless self-attention', plain_n, plain_sep),
                     ('trained Q/K/V self-attention', attn_n, attn_sep)]:
    print(f'{name:<40}{f"{n}/{NT}":>9}{n/NT:>8.1%}{sep:>15.6f}')
print(f'{"(majority class)":<40}{"":>9}{majority:>8.1%}')

### Read this table carefully — one row is a definition, and one comparison fails

**The static row's cosine of 1.000000 is not a measurement.** One word means one
vector, so the mean *drive* vector over car documents and over hardware documents are
the identical vector, and the cosine of a thing with itself is 1. The classifier above
it can therefore only guess the majority class. This is lesson 072's finding arriving
as a number on a task.

**And the honest part: accuracy did not separate the two attention rows.** They get
the same count. Telling a car post from a hardware post is a coarse topic distinction
and averaging the context words already settles it, so accuracy saturates — and a test
split this small could not resolve a small difference even if one existed. *Anyone
reporting only accuracy here would conclude Q/K/V buys nothing.*

The claim being tested is about the **representation**, so it has to be measured on
the representation. Parameterless attention leaves the two sense-means at about
`0.91` — off 1.0, but still nearly one vector. Trained Q/K/V takes them to roughly
`-0.97`, pointing opposite ways.

Treat that magnitude with care, though: a two-class linear readout has every incentive
to push the class means apart, so **the interesting fact is not how far they went but
that they can move at all.** The static embedding is pinned at exactly 1.000000 by
construction, and no training in any layer above it can ever change that number.

In [ ]:
# What did 'drive' learn to attend to? Look at one document from each class.
shown = {0: None, 1: None}
for i in train_idx:
    ids, pos, label, ctx = examples[i]
    if shown[label] is None and len(ctx) >= 8:
        *_, a, _, _ = forward(Wq, Wk, Wv, Wo, b, ids, pos)
        shown[label] = (ctx, a)
    if all(v is not None for v in shown.values()):
        break

for label, (ctx, a) in shown.items():
    top = np.argsort(-a)[:5]
    print(f'{CATS[label]:<28}', ', '.join(f'{ctx[j]} {a[j]:.2f}' for j in top))

## Part E — Only the product `W_q @ W_k.T` ever reaches the scores

Since `Q = X W_q` and `K = X W_k`, the score matrix is `X (W_q W_k.T) X.T`. So a pair
of `d × d_k` matrices is parameterising a **single `d × d` matrix**, and that matrix
has rank at most `d_k`.

In [ ]:
d_model, d_k, n_heads = 512, 64, 8
Wq_big = np.random.randn(d_model, d_k) / np.sqrt(d_model)
Wk_big = np.random.randn(d_model, d_k) / np.sqrt(d_model)
M  = Wq_big @ Wk_big.T
Xs = np.random.randn(12, d_model)

direct = (Xs @ Wq_big) @ (Xs @ Wk_big).T
via_M  = Xs @ M @ Xs.T
print('max |Q K.T - X M X.T|          ', f'{np.abs(direct - via_M).max():.1e}')

# Any other factorisation of the SAME product gives identical attention.
Um, Sm, Vm = np.linalg.svd(M)
half  = Um[:, :d_k] * np.sqrt(Sm[:d_k])
other = Vm[:d_k].T  * np.sqrt(Sm[:d_k])
refactored = (Xs @ half) @ (Xs @ other).T
print('max |Q K.T - refactored|       ', f'{np.abs(direct - refactored).max():.1e}')
print('rank(W_q @ W_k.T)              ', int((Sm > 1e-10).sum()), 'of', d_model)

print('\nQ/K/V parameters at d_model=512:', f'{3*d_model**2 + 3*d_model:,}')
print('parameterless version:          ', 0)
print(f'as a share of a 37,000 x 512 embedding table: '
      f'{(3*d_model**2 + 3*d_model) / (37_000*512):.1%}')

**A single head cannot express an arbitrary interaction between 512 dimensions — it is
confined to a 64-dimensional slice of them.** That restriction is deliberate, and it is
the argument for running 8 heads of 64 rather than one head of 512: eight rank-64
views, each free to specialise. Lessons 075 and 076 develop it.

And note the last line. One attention block is about **4% of the embedding table**, so
lesson 071's finding that the transformer inverts the usual parameter split is a fact
about **depth** — twelve stacked blocks plus their feed-forward layers — not about
attention being expensive.

## Exercises

1. **Break the symmetry claim.** Try to find any `X` for which `X @ X.T` is *not*
   symmetric. Then prove to yourself why you cannot.
2. **The order experiment.** Reverse the words in a sentence and confirm the set of
   output vectors is unchanged (just reordered). This is the debt lesson 077 repays.
3. **Sweep the window.** Re-run Part D with `WINDOW = 2` and `WINDOW = 20`. Does the
   sense cosine still separate? What happens to the number of usable examples?
4. **A harder task.** Replace the two categories with a pair that a bag of context
   words *cannot* separate easily, and see whether accuracy starts distinguishing
   trained attention from the parameterless version.
5. **Value matters too.** Set `Wv` to the identity (so values are raw embeddings) and
   train only `Wq`, `Wk`. How much of the separation survives?